In [2]:
import pandas as pd 
from IPython.display import display

# 读取数据
movies = pd.read_csv('movies.csv')
display(movies.head(10))

# 创建 genre 矩阵（one-hot 编码）
genre_matrix = movies['genres'].str.get_dummies(sep='|')

# 显示矩阵
display(genre_matrix.head())

# 检查类型
print(f"genre_matrix 类型: {type(genre_matrix)}")
print(f"genre_matrix 形状: {genre_matrix.shape}")

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


genre_matrix 类型: <class 'pandas.core.frame.DataFrame'>
genre_matrix 形状: (9125, 20)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim=cosine_similarity(genre_matrix)
print("相似度矩阵形状:",cosine_sim.shape)

相似度矩阵形状: (9125, 9125)


In [4]:
def get_recommendations(title):
    try:
        idx = movies[movies['title'] == title].index[0]
    except:
        return "电影库没找到这部电影，请检查拼写（包含年份）"
    
    # 获取相似度分数
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]  # 排除自己，取前5个
    
    # 提取电影索引和分数
    movie_indices = [i[0] for i in sim_scores]
    movie_scores = [i[1] for i in sim_scores]
    
    # 创建结果DataFrame
    recommendations = movies['title'].iloc[movie_indices].reset_index(drop=True)
    scores_df = pd.DataFrame(movie_scores, columns=['cosine_similarity'])
    
    # 合并显示
    result = pd.concat([recommendations, scores_df], axis=1)
    result.columns = ['推荐电影', '相似度分数']
    
    return result

In [5]:
test_movie=movies['title'].iloc[9]
print(f"测试电影:{test_movie}")
recommendations=get_recommendations(test_movie)
print(recommendations)

测试电影:GoldenEye (1995)
                        推荐电影  相似度分数
0        Broken Arrow (1996)    1.0
1         Cliffhanger (1993)    1.0
2  Executive Decision (1996)    1.0
3  Surviving the Game (1994)    1.0
4           Rock, The (1996)    1.0


In [6]:
print(movies[movies['title']==test_movie])

   movieId             title                     genres
9       10  GoldenEye (1995)  Action|Adventure|Thriller


In [7]:
display(movies.head(491))

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
486,542,Son in Law (1993),Comedy|Drama|Romance
487,543,So I Married an Axe Murderer (1993),Comedy|Romance|Thriller
488,544,Striking Distance (1993),Action|Crime
489,546,Super Mario Bros. (1993),Action|Adventure|Children|Comedy|Fantasy|Sci-Fi


In [8]:
#基于协同过滤的个性化推荐

In [9]:
import pandas as pd
from IPython.display import display

In [10]:
#读取文件
ratings=pd.read_csv('ratings.csv')
display(ratings.head(20))

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
5,1,1263,2.0,1260759151
6,1,1287,2.0,1260759187
7,1,1293,2.0,1260759148
8,1,1339,3.5,1260759125
9,1,1343,2.0,1260759131


In [11]:
#将表格转换成矩阵
user_movie_matrix=ratings.pivot(index='userId',columns='movieId',values='rating')

In [12]:
#矩阵的列是电影（有至少10个评分）
user_movie_matrix=user_movie_matrix.dropna(thresh=10,axis=1)

In [13]:
#矩阵的行是用户（有看过至少20部）
user_movie_matrix=user_movie_matrix.dropna(thresh=20,axis=0)

In [14]:
user_movie_matrix_filled=user_movie_matrix.fillna(0)
print("过滤后的矩阵形状",user_movie_matrix_filled.shape)
print("前五行数据",user_movie_matrix_filled.head(5))

过滤后的矩阵形状 (646, 2245)
前五行数据 movieId  1       2       3       4       5       6       7       9       \
userId                                                                    
1           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
5           0.0     0.0     4.0     0.0     0.0     0.0     0.0     0.0   

movieId  10      11      ...  119145  122882  122886  122892  122900  122904  \
userId                   ...                                                   
1           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
2           4.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
4           4.0     0.0  ...     0.0     0.0   

In [15]:
#计算用户相似度
from sklearn.metrics.pairwise import cosine_similarity
user_sim=cosine_similarity(user_movie_matrix_filled)
#转换成Dataframe，给定标签便于索引
user_sim_df=pd.DataFrame(user_sim,
                         index=user_movie_matrix.index,
                         columns=user_movie_matrix.index)
print("用户相似度矩阵（前五名用户）:")
print(user_sim_df.head(5))

用户相似度矩阵（前五名用户）:
userId       1         2         3         4         5         6         7    \
userId                                                                         
1       1.000000  0.000000  0.000000  0.076850  0.016818  0.000000  0.084483   
2       0.000000  1.000000  0.129935  0.122597  0.103646  0.000000  0.214505   
3       0.000000  0.129935  1.000000  0.088057  0.158407  0.063445  0.162889   
4       0.076850  0.122597  0.088057  1.000000  0.134801  0.082180  0.332262   
5       0.016818  0.103646  0.158407  0.134801  1.000000  0.063796  0.096572   

userId       8         9         10   ...       660       661       662  \
userId                                ...                                 
1       0.000000  0.012930  0.000000  ...  0.000000  0.032313  0.000000   
2       0.114279  0.114103  0.046108  ...  0.018829  0.024796  0.485901   
3       0.263626  0.141532  0.127906  ...  0.056189  0.088588  0.171555   
4       0.198979  0.031597  0.151027  ...  0.060

In [16]:
def get_user_recommendations(user_id, top_n=5):
    # 1. 找到和该用户最像的前 10 个人 (用方括号，变量名对齐)
    similar_users = user_sim_df[user_id].sort_values(ascending=False)[1:11].index
    
    # 2. 看看这些“邻居”都评价过哪些电影 
    neighbor_ratings = ratings[ratings['userId'].isin(similar_users)]
    
    # 3. 排除掉该用户已经看过的电影
    user_watched = ratings[ratings['userId'] == user_id]['movieId']
    recommendations = neighbor_ratings[~neighbor_ratings['movieId'].isin(user_watched)]
    
    # 4. 计算这些电影在邻居中的平均分，并排序
    res = recommendations.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(top_n)
    
    # 5. 联动 movies 表展示结果
    return movies[movies['movieId'].isin(res.index)][['title', 'genres']]

In [17]:
print(f"为用户{user_movie_matrix.index[0]}推荐的电影列表:")
get_user_recommendations(user_movie_matrix.index[0])

为用户1推荐的电影列表:


,title,genres
64,Friday (1995),Comedy
1636,Blue Velvet (1986),Drama|Mystery|Thriller
1715,Rosemary's Baby (1968),Drama|Horror|Thriller
3871,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
5912,Ali G Indahouse (2002),Comedy


In [19]:
#去中心化
#计算每个用户的平均分
user_mean=user_movie_matrix.mean(axis=1)
#去中心化
user_movie_matrix_centered=user_movie_matrix.sub(user_mean,axis=0)
#以0代替nan
user_movie_matrix_final=user_movie_matrix_centered.fillna(0)
#打印矩阵并查看
print(user_movie_matrix_final.head())

movieId  1       2       3       4       5       6       7       9       \
userId                                                                    
1           0.0     0.0    0.00     0.0     0.0     0.0     0.0     0.0   
2           0.0     0.0    0.00     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0    0.00     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0    0.00     0.0     0.0     0.0     0.0     0.0   
5           0.0     0.0    0.09     0.0     0.0     0.0     0.0     0.0   

movieId    10      11      ...  119145  122882  122886  122892  122900  \
userId                     ...                                           
1        0.000000     0.0  ...     0.0     0.0     0.0     0.0     0.0   
2        0.513158     0.0  ...     0.0     0.0     0.0     0.0     0.0   
3        0.000000     0.0  ...     0.0     0.0     0.0     0.0     0.0   
4       -0.404255     0.0  ...     0.0     0.0     0.0     0.0     0.0   
5        0.000000     0.0  ...

In [22]:
#计算Pearson相关系数
from sklearn.metrics.pairwise import cosine_similarity
#计算更新后的相似度
user_sim_improved=cosine_similarity(user_movie_matrix_final)
#转换成DataFrame
user_sim_df_improved=pd.DataFrame(user_sim_improved,
                               index=user_movie_matrix.index,
                               columns=user_movie_matrix.index)
print(user_sim_df_improved.head())

userId       1         2         3         4         5         6         7    \
userId                                                                         
1       1.000000  0.000000  0.000000  0.003129 -0.002274  0.000000 -0.070957   
2       0.000000  1.000000 -0.001473 -0.006407  0.012639  0.000000  0.043199   
3       0.000000 -0.001473  1.000000  0.020438 -0.026273 -0.063670  0.056328   
4       0.003129 -0.006407  0.020438  1.000000  0.012255  0.019583  0.061403   
5      -0.002274  0.012639 -0.026273  0.012255  1.000000 -0.005843 -0.014991   

userId       8         9         10   ...       660       661       662  \
userId                                ...                                 
1       0.000000  0.043420  0.000000  ...  0.000000 -0.012833  0.000000   
2       0.020426  0.010418 -0.010356  ...  0.038012  0.005928 -0.019976   
3       0.026068 -0.037669  0.041729  ... -0.002878  0.003360  0.048746   
4       0.059182 -0.011142  0.002690  ... -0.040835 -0.000334  0

In [35]:
#编写混合推荐
def get_hybird_recommendations(user_id,top_n=5):
    #1.用户看过超过20部
    if user_id in user_sim_df_improved.index:
        #1.找最像的10名用户
        similar_users=user_sim_df_improved[user_id].sort_values(ascending=False)[1:11].index
        #2.找10名用户看过但该用户没看过的电影
        user_watched=ratings[ratings['userId']==user_id]['movieId']
        neighbor_ratings=ratings[ratings['userId'].isin(similar_users)]
        recommendations=neighbor_ratings[~neighbor_ratings['movieId'].isin(user_watched)]
        #3.按平均排序
        res=recommendations.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(top_n)
        print(f">>>为用户{user_id}匹配喜好口味:")
        return movies[movies['movieId'].isin(res.index)][['title','genres']]
    #2.用户不在矩阵中
    else:
        #1.按热度排序推荐
        popular_movies=ratings.groupby('movieId')['rating'].count().sort_values(ascending=False).head(top_n)
        print(f">>>识别为新用户{user_id},正在进行热度补偿推荐:")
        return movies[movies['movieId'].isin(popular_movies.index)][['title','genres']]

In [41]:
#测试1：1号活跃用户
print("为活跃用户生成的个性化菜单")
res_active=get_hybird_recommendations(1)
display(res_active)

为活跃用户生成的个性化菜单
>>>为用户1匹配喜好口味:


,title,genres
2,Grumpier Old Men (1995),Comedy|Romance
1583,Dune (1984),Adventure|Sci-Fi
1590,Saving Private Ryan (1998),Action|Drama|War
4395,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy
5026,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy


In [39]:
#测试2：模拟一个新用户9999（不在矩阵中）
print("为新用户生成的推荐菜单")
res_new=get_hybird_recommendations(9999)
display(res_new)

为新用户生成的推荐菜单
>>>识别为新用户9999,正在进行热度补偿推荐:


,title,genres
232,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
266,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
284,"Shawshank Redemption, The (1994)",Crime|Drama
321,Forrest Gump (1994),Comedy|Drama|Romance|War
525,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
